In [1]:
import pandas as pd
import glob
import os
import numpy as np
import math

def sample_8_percent_precise():
    """精确的8%抽样方法"""
    
    current_dir = os.getcwd()
    
    # 获取原始数据文件
    parquet_files = []
    for f in glob.glob(os.path.join(current_dir, "fhvhv_tripdata_2024-*.parquet")):
        parquet_files.append(f)
    
    if not parquet_files:
        print("没有找到2024年的行程数据文件")
        return
    
    print(f"找到 {len(parquet_files)} 个月份的数据文件")
    print("开始8%精确抽样处理...")
    print("=" * 60)
    
    all_samples = []
    monthly_stats = []
    total_original = 0
    total_sampled = 0
    
    for i, file_path in enumerate(parquet_files, 1):
        filename = os.path.basename(file_path)
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        month = filename.split('_')[-1].replace('.parquet', '')
        
        print(f"[{i}/{len(parquet_files)}] 处理 {month} 月:")
        print(f"   文件: {filename}")
        print(f"   大小: {file_size_mb:,.1f} MB")
        
        try:
            # 读取文件
            df = pd.read_parquet(file_path)
            original_rows = len(df)
            total_original += original_rows
            
            print(f"   总行数: {original_rows:,}")
            
            # 计算8%抽样数量
            target_sample = max(1, math.ceil(original_rows * 0.08))
            print(f"   目标抽样: {target_sample:,} 行 (8%)")
            
            if original_rows <= 12:  # 如果行数很少，全部保留
                sampled = df
                print(f"   → 行数较少，保留全部")
            else:
                # 使用高效抽样
                np.random.seed(2025 + i)
                
                if original_rows <= 5000000:  # 500万行以内
                    sampled = df.sample(frac=0.08, random_state=42, replace=False)
                else:
                    # 使用numpy抽样
                    sample_indices = np.random.choice(original_rows, size=target_sample, replace=False)
                    sampled = df.iloc[sample_indices].copy()
            
            sampled_rows = len(sampled)
            total_sampled += sampled_rows
            sample_percent = (sampled_rows / original_rows * 100) if original_rows > 0 else 100
            
            print(f"   ✅ 实际抽样: {sampled_rows:,} 行 ({sample_percent:.1f}%)")
            
            all_samples.append(sampled)
            
            monthly_stats.append({
                '月份': month,
                '原始行数': original_rows,
                '抽样行数': sampled_rows,
                '百分比': sample_percent
            })
            
            # 释放内存
            del df
            
        except MemoryError:
            print(f"   ⚠ 内存不足，使用系统抽样...")
            
            try:
                # 系统抽样：每12-13行取1行
                step = max(1, int(1 / 0.08))  # 约12.5
                
                # 读取前N行进行系统抽样
                read_rows = min(original_rows, 1000000)  # 最多读100万行
                df_sample = pd.read_parquet(file_path, nrows=read_rows)
                
                if len(df_sample) > 0:
                    # 系统抽样
                    sampled = df_sample.iloc[::step].copy()
                    
                    # 确保大约8%
                    if len(sampled) > read_rows * 0.10:  # 如果超过10%，调整
                        target = int(read_rows * 0.08)
                        sampled = sampled.head(target)
                    elif len(sampled) < read_rows * 0.06:  # 如果少于6%，调整
                        target = int(read_rows * 0.08)
                        additional = target - len(sampled)
                        if additional > 0:
                            remaining = df_sample.drop(sampled.index)
                            extra = remaining.sample(n=min(additional, len(remaining)), random_state=42)
                            sampled = pd.concat([sampled, extra])
                    
                    all_samples.append(sampled)
                    total_sampled += len(sampled)
                    
                    monthly_stats.append({
                        '月份': month,
                        '原始行数': f"{read_rows:,}(部分)",
                        '抽样行数': len(sampled),
                        '百分比': (len(sampled)/read_rows*100) if read_rows > 0 else 0
                    })
                    
                    print(f"   ✅ 系统抽样: {len(sampled):,} 行")
                
            except Exception as e:
                print(f"   ❌ 系统抽样失败: {e}")
                
        except Exception as e:
            print(f"   ❌ 处理失败: {e}")
    
    if all_samples:
        print(f"\n{'='*60}")
        print("合并所有抽样数据...")
        
        merged_df = pd.concat(all_samples, ignore_index=True)
        
        output_file = "2025_8percent_sampled.parquet"
        output_path = os.path.join(current_dir, output_file)
        
        print(f"保存到: {output_path}")
        print("正在保存，请稍候...")
        
        # 保存文件
        merged_df.to_parquet(output_path, index=False, compression='snappy')
        
        # 统计信息
        output_size_mb = os.path.getsize(output_path) / (1024 * 1024)
        estimated_original_size = sum(os.path.getsize(f) for f in parquet_files) / (1024 * 1024)
        
        print(f"\n{'='*60}")
        print("✅ 8%抽样完成!")
        print(f"{'='*60}")
        
        print(f"📊 总体统计:")
        print(f"  原始文件总数: {len(parquet_files)}")
        print(f"  原始总大小: {estimated_original_size:,.1f} MB")
        print(f"  原始总行数: {total_original:,}")
        print(f"  抽样总行数: {total_sampled:,}")
        
        if total_original > 0:
            overall_percent = (total_sampled / total_original * 100)
            print(f"  总体抽样比例: {overall_percent:.1f}%")
            size_reduction = ((estimated_original_size - output_size_mb) / estimated_original_size * 100)
            print(f"  大小缩减: {size_reduction:.1f}%")
            
            # 与25%对比
            estimated_25pct_size = estimated_original_size * 0.25
            reduction_vs_25pct = ((estimated_25pct_size - output_size_mb) / estimated_25pct_size * 100)
            print(f"  相比25%抽样减少: {reduction_vs_25pct:.1f}%")
        
        print(f"\n📁 输出文件:")
        print(f"  名称: {output_file}")
        print(f"  大小: {output_size_mb:,.1f} MB (~{output_size_mb/1024:.2f} GB)")
        print(f"  行数: {len(merged_df):,}")
        print(f"  列数: {len(merged_df.columns)}")
        
        # 月度详细统计
        print(f"\n📅 月度统计:")
        print("-" * 55)
        print(f"{'月份':<8} {'原始行数':>12} {'抽样行数':>12} {'比例':>8}")
        print("-" * 55)
        
        for stat in monthly_stats:
            if isinstance(stat['原始行数'], (int, float)):
                print(f"{stat['月份']:<8} {stat['原始行数']:>12,} {stat['抽样行数']:>12,} {stat['百分比']:>7.1f}%")
            else:
                print(f"{stat['月份']:<8} {stat['原始行数']:>12} {stat['抽样行数']:>12,} {'N/A':>8}")
        
        print("-" * 55)
        
        # 数据预览
        print(f"\n🔍 数据预览 (前5行):")
        print(merged_df.head())
        
        return merged_df
    
    return None

# 运行8%抽样
result = sample_8_percent_precise()

找到 12 个月份的数据文件
开始8%精确抽样处理...
[1/12] 处理 2024-01 月:
   文件: fhvhv_tripdata_2024-01.parquet
   大小: 450.9 MB
   总行数: 19,663,930
   目标抽样: 1,573,115 行 (8%)
   ✅ 实际抽样: 1,573,115 行 (8.0%)
[2/12] 处理 2024-02 月:
   文件: fhvhv_tripdata_2024-02.parquet
   大小: 441.1 MB
   总行数: 19,359,148
   目标抽样: 1,548,732 行 (8%)
   ✅ 实际抽样: 1,548,732 行 (8.0%)
[3/12] 处理 2024-03 月:
   文件: fhvhv_tripdata_2024-03.parquet
   大小: 484.4 MB
   总行数: 21,280,788
   目标抽样: 1,702,464 行 (8%)
   ✅ 实际抽样: 1,702,464 行 (8.0%)
[4/12] 处理 2024-04 月:
   文件: fhvhv_tripdata_2024-04.parquet
   大小: 454.1 MB
   总行数: 19,733,038
   目标抽样: 1,578,644 行 (8%)
   ✅ 实际抽样: 1,578,644 行 (8.0%)
[5/12] 处理 2024-05 月:
   文件: fhvhv_tripdata_2024-05.parquet
   大小: 475.5 MB
   总行数: 20,704,538
   目标抽样: 1,656,364 行 (8%)
   ✅ 实际抽样: 1,656,364 行 (8.0%)
[6/12] 处理 2024-06 月:
   文件: fhvhv_tripdata_2024-06.parquet
   大小: 462.0 MB
   总行数: 20,123,226
   目标抽样: 1,609,859 行 (8%)
   ✅ 实际抽样: 1,609,859 行 (8.0%)
[7/12] 处理 2024-07 月:
   文件: fhvhv_tripdata_2024-07.parquet
   大小: 446.7